# Inspect Delta Tables

Use this notebook to inspect local Delta Lake outputs created by the streaming jobs. Start JupyterLab from the repository root and select the `Crypto Market Intelligence (.venv)` kernel.

In [ ]:
from pathlib import Path
import os
import sys

repo_root = Path.cwd()
if not (repo_root / "streaming").exists() and (repo_root.parent / "streaming").exists():
    repo_root = repo_root.parent

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repository root: {repo_root}")

In [ ]:
from streaming.spark_session import create_spark_session

spark = create_spark_session("inspect-local-delta-tables")
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark version: {spark.version}")

In [ ]:
DELTA_TABLES = {
    "bronze_trades": "./storage/bronze/bronze_market_trades_raw",
    "bronze_klines": "./storage/bronze/bronze_market_klines_raw",
    "bronze_tickers": "./storage/bronze/bronze_market_tickers_raw",
    "bronze_invalid_events": "./storage/bronze/bronze_market_invalid_events",
    "silver_trades": "./storage/silver/silver_market_trades",
    "silver_klines": "./storage/silver/silver_market_klines",
    "silver_tickers": "./storage/silver/silver_market_tickers",
    "gold_ohlc_1min": "./storage/gold/gold_symbol_1min_ohlc",
    "gold_trade_summary_5min": "./storage/gold/gold_symbol_5min_trade_summary",
    "gold_volatility_5min": "./storage/gold/gold_symbol_5min_volatility",
    "gold_volume_spike_signals": "./storage/gold/gold_volume_spike_signals",
    "gold_price_movement_alerts": "./storage/gold/gold_price_movement_alerts",
    "gold_watchlist_summary": "./storage/gold/gold_market_watchlist_summary",
}

for name, path in DELTA_TABLES.items():
    print(f"{name}: {path}")

In [ ]:
import pandas as pd
from pyspark.sql import functions as F


def delta_exists(path: str) -> bool:
    return (Path(path) / "_delta_log").exists()


def table_status() -> pd.DataFrame:
    rows = []
    for name, path in DELTA_TABLES.items():
        delta_log_exists = delta_exists(path)
        rows.append(
            {
                "table": name,
                "path": path,
                "delta_log_exists": delta_log_exists,
                "path_exists": Path(path).exists(),
            }
        )
    return pd.DataFrame(rows)


def inspect_delta_table(table_name: str, limit: int = 20, order_by: str | None = None):
    path = DELTA_TABLES[table_name]
    if not delta_exists(path):
        print(f"No Delta table found at {path}")
        print("Start the producer and the matching Bronze stream, then rerun this cell.")
        return None

    df = spark.read.format("delta").load(path)
    print(f"Path: {path}")
    print("Schema:")
    df.printSchema()
    print(f"Rows: {df.count()}")

    sample_df = df
    if order_by and order_by in df.columns:
        sample_df = sample_df.orderBy(F.col(order_by).desc())

    display(sample_df.limit(limit).toPandas())
    return df

In [ ]:
table_status()

In [ ]:
table_name = "bronze_trades"
df = inspect_delta_table(table_name, limit=20, order_by="kafka_timestamp")

In [ ]:
if df is not None:
    df.createOrReplaceTempView("selected_delta_table")
    spark.sql(
        """
        select
            topic,
            key as symbol,
            count(*) as row_count,
            min(kafka_timestamp) as first_kafka_timestamp,
            max(kafka_timestamp) as latest_kafka_timestamp
        from selected_delta_table
        group by topic, key
        order by row_count desc
        """
    ).toPandas()
else:
    print("No selected table loaded yet.")

In [ ]:
path = DELTA_TABLES[table_name]
if delta_exists(path):
    abs_path = str(Path(path).resolve())
    display(spark.sql(f"DESCRIBE DETAIL delta.`{abs_path}`").toPandas())
else:
    print(f"No Delta metadata found for {path}")

Stop the Spark session when you are done with the notebook.

In [ ]:
# spark.stop()

In [ ]:
from pathlib import Path

tables = {
    "bronze_market_trades_raw": "./storage/bronze/bronze_market_trades_raw",
    "bronze_market_klines_raw": "./storage/bronze/bronze_market_klines_raw",
    "bronze_market_tickers_raw": "./storage/bronze/bronze_market_tickers_raw",
    "bronze_market_invalid_events": "./storage/bronze/bronze_market_invalid_events",
    "silver_market_trades": "./storage/silver/silver_market_trades",
    "silver_market_klines": "./storage/silver/silver_market_klines",
    "silver_market_tickers": "./storage/silver/silver_market_tickers",
    "gold_symbol_1min_ohlc": "./storage/gold/gold_symbol_1min_ohlc",
    "gold_symbol_5min_trade_summary": "./storage/gold/gold_symbol_5min_trade_summary",
    "gold_symbol_5min_volatility": "./storage/gold/gold_symbol_5min_volatility",
    "gold_volume_spike_signals": "./storage/gold/gold_volume_spike_signals",
    "gold_price_movement_alerts": "./storage/gold/gold_price_movement_alerts",
    "gold_market_watchlist_summary": "./storage/gold/gold_market_watchlist_summary",
}

spark.sql("CREATE DATABASE IF NOT EXISTS crypto_market")
spark.sql("USE crypto_market")

for table_name, path in tables.items():
    abs_path = str(Path(path).resolve())
    if (Path(abs_path) / "_delta_log").exists():
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {table_name}
            USING DELTA
            LOCATION '{abs_path}'
        """)
        print(f"registered {table_name}")
    else:
        print(f"skipped {table_name}: no Delta log yet")


In [ ]:
spark.sql("""
SELECT *
FROM crypto_market.silver_market_tickers
""").toPandas()


## Gold Tables

These queries inspect the business-level outputs used for dashboarding and alert publishing.

In [ ]:
gold_table_names = [
    "gold_symbol_1min_ohlc",
    "gold_symbol_5min_trade_summary",
    "gold_symbol_5min_volatility",
    "gold_volume_spike_signals",
    "gold_price_movement_alerts",
    "gold_market_watchlist_summary",
]

rows = []
for table_name in gold_table_names:
    df = spark.table(f"crypto_market.{table_name}")
    rows.append({"table": table_name, "rows": df.count(), "columns": len(df.columns)})

pd.DataFrame(rows).sort_values("table")

In [ ]:
spark.sql("""
SELECT *
FROM crypto_market.gold_market_watchlist_summary
ORDER BY last_updated_time DESC, symbol
LIMIT 30
""").toPandas()

In [ ]:
spark.sql("""
SELECT *
FROM crypto_market.gold_volume_spike_signals
ORDER BY window_end DESC, volume_spike_ratio DESC
LIMIT 30
""").toPandas()

In [ ]:
spark.sql("""
SELECT *
FROM crypto_market.gold_price_movement_alerts
ORDER BY window_end DESC, ABS(price_change_pct) DESC
LIMIT 30
""").toPandas()

## Alert Kafka Topic

The alert layer publishes JSON events to Kafka rather than writing a separate Delta table. This batch read inspects the current contents of `market.signals.alerts`.

In [ ]:
from pyspark.sql import functions as F

alert_topic = "market.signals.alerts"
alert_messages = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", alert_topic)
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
    .select(
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.col("key").cast("string").alias("symbol_key"),
        F.col("value").cast("string").alias("alert_json"),
    )
)

print(f"Alert messages: {alert_messages.count()}")
display(alert_messages.orderBy(F.col("offset").desc()).limit(30).toPandas())